In [ ]:
import sys, glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from gtsfm.common.depth_provider import DepthProvider

SEQ = 'office0'
DATA_ROOT = ''
DATA = f'{DATA_ROOT}/{SEQ}'
DEPTH_DIR = f'{DATA}/results'
DEPTH_SCALE = 6553.5
FRAME_IDX = 0

rgb_paths = sorted(glob.glob(f'{DATA}/results/frame*.jpg'))
depth_paths = sorted(glob.glob(f'{DEPTH_DIR}/depth*.png'))

rgb = cv2.cvtColor(cv2.imread(rgb_paths[FRAME_IDX]), cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
depth_m = cv2.imread(depth_paths[FRAME_IDX], cv2.IMREAD_UNCHANGED).astype(np.float64) / DEPTH_SCALE

kps = cv2.SIFT_create().detect(gray)
uvs = np.array([kp.pt for kp in kps])  # (N,2): col=u, row=v
print(f'SIFT keypoints: {len(uvs)}')

In [ ]:
PATCH_RADIUS = 5
GAP_THRESH = 0.15
AMBIGUITY_THRESH = 0.20

dp = DepthProvider(
    depth_map_dir=DEPTH_DIR,
    image_fnames={0: rgb_paths[FRAME_IDX]},
    depth_min=0.1, depth_max=10.0,
    depth_scale=DEPTH_SCALE,
    compute_hypotheses=True,
    patch_radius=PATCH_RADIUS,
    gap_thresh=GAP_THRESH,
    ambiguity_thresh=AMBIGUITY_THRESH,
)
samples = [dp.get_depth(0, u, v) for u, v in uvs]
scores = np.array([s.score if s else 0.0 for s in samples])
ambiguous = np.array([s.ambiguous if s else False for s in samples])
valid = np.array([s is not None for s in samples])
print(f'Valid: {valid.sum()} / {len(uvs)},  Ambiguous: {ambiguous.sum()} ({100*ambiguous.mean():.3f}%)')

In [ ]:
# Viz 1: score heatmap on depth map + flagged kps on RGB
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(depth_m, cmap='plasma', vmin=0, vmax=5)
sc = axes[0].scatter(uvs[valid,0], uvs[valid,1], c=scores[valid], cmap='hot', s=8, vmin=0, vmax=0.5)
plt.colorbar(sc, ax=axes[0], label='ambiguity score')
axes[0].set_title(f'Scores (R={PATCH_RADIUS}, gap={GAP_THRESH})')
axes[1].imshow(rgb)
axes[1].scatter(uvs[~ambiguous,0], uvs[~ambiguous,1], c='cyan', s=5, label='unimodal')
axes[1].scatter(uvs[ambiguous, 0], uvs[ambiguous, 1], c='red',  s=20, label='ambiguous')
axes[1].legend()
axes[1].set_title(f'Flagged: {ambiguous.sum()} / {len(uvs)}')
plt.tight_layout(); plt.show()

In [ ]:
# Viz 2: histogram of kp distance to nearest depth edge
depth_u8 = np.clip(depth_m / 5.0 * 255, 0, 255).astype(np.uint8)
edges    = cv2.Canny(depth_u8, 10, 30)
dist_map = cv2.distanceTransform((255 - edges), cv2.DIST_L2, 5)
kp_dists = np.array([
    dist_map[int(np.clip(round(v), 0, dist_map.shape[0]-1)),
             int(np.clip(round(u), 0, dist_map.shape[1]-1))]
    for u, v in uvs
])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(edges, cmap='gray')
axes[0].scatter(uvs[:,0], uvs[:,1], c='red', s=4, alpha=0.5)
axes[0].set_title('Depth edges + SIFT kps')
axes[1].hist(kp_dists, bins=50, color='steelblue', edgecolor='k')
axes[1].axvline(PATCH_RADIUS, color='red', linestyle='--', label=f'patch_radius={PATCH_RADIUS}')
axes[1].set_xlabel('Distance to nearest depth edge (px)')
axes[1].set_ylabel('# keypoints')
axes[1].set_title('How close do SIFT kps land to depth edges?')
axes[1].legend()
print(f'Kps within {PATCH_RADIUS}px of a depth edge: {100*(kp_dists <= PATCH_RADIUS).mean():.1f}%')
plt.tight_layout(); plt.show()

In [ ]:
# Viz 3: % flagged vs ambiguity_thresh
# Re-run with thresh=0 to collect raw scores, then threshold in numpy
dp_raw = DepthProvider(
    depth_map_dir=DEPTH_DIR,
    image_fnames={0: rgb_paths[FRAME_IDX]},
    depth_min=0.1, depth_max=10.0,
    depth_scale=DEPTH_SCALE,
    compute_hypotheses=True,
    patch_radius=PATCH_RADIUS,
    gap_thresh=0.0,
    ambiguity_thresh=0.0,
)
raw_scores  = np.array([s.score if s else 0.0 for s in [dp_raw.get_depth(0, u, v) for u, v in uvs]])
thresholds  = np.linspace(0.0, 0.5, 50)
pct_flagged = [100.0 * (raw_scores >= t).mean() for t in thresholds]
plt.figure(figsize=(7, 4))
plt.plot(thresholds, pct_flagged)
plt.axvline(AMBIGUITY_THRESH, color='red', linestyle='--', label=f'current thresh={AMBIGUITY_THRESH}')
plt.xlabel('ambiguity_thresh')
plt.ylabel('% keypoints flagged')
plt.title('Sensitivity of ambiguous fraction to threshold')
plt.legend(); plt.tight_layout(); plt.show()